**Transform Orders Data-String To JSON Object**

In [0]:
%python
dfOrders=spark.table("gizmobox_gr.bronze.py_orders");
display(dfOrders)

**Pre-Process JSON String To Fix The Data Quality Issues**

In [0]:
%python
from pyspark.sql import functions as f
dfOrders_DataQuality=dfOrders.select(
    "value",
    f.regexp_replace("value", '"order_date": (\\d{4}-\\d{2}-\\d{2})', '"order_date": "$1"').alias("fixed_value")
)
display(dfOrders_DataQuality)

**Transform JSON String To JSON Object**

In [0]:
%python
from pyspark.sql import functions as f
dfOrders_jsonSchema=(
dfOrders_DataQuality.select(
    f.schema_of_json(dfOrders_DataQuality.fixed_value).alias("schema")
    )
)
display(dfOrders_jsonSchema)

In [0]:
%python
from pyspark.sql import functions as f
dfOrders_jsonObject=dfOrders_DataQuality.select(
    f.from_json("fixed_value", 'STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>').alias('jsonObject')
)
display(dfOrders_jsonObject)

**Write Transformed Data To Silver Schema**

In [0]:
%python
dfOrders_jsonObject.writeTo("gizmobox_gr.silver.py_orders_json").createOrReplace()
display(spark.table("gizmobox_gr.silver.py_orders_json"))